In [1]:
#Setup paths + imports
%cd ../backend/data
import sys
import os
sys.path.append(os.path.abspath('.'))

from db import DBSession, Track, init_db
from ingest_relational_DB import ingest_h5_dir, ingest_lastfm_json_subset, ingest_musixmatch_sqlite, ingest_genres
from pathlib import Path
import sqlite3

print("✅ Working dir:", os.getcwd())
print("✅ DB path:", "music_relational.db")

# Paths from backend/data/
H5_DIR = "../../../Data/MillionSongSubset"
LASTFM_JSON_DIR = "../../../Data/lastfm_subset"
MXM_DB = "../../../Data/mxm_dataset.db"
LASTFM_MAP = "../../../Data/msd_lastfm_map.cls"
DB_URL = "sqlite:///music_relational.db"

# Init DB
init_db(DB_URL)


/Users/dc1408/side_hustle/EchoAgent/backend/data
✅ Working dir: /Users/dc1408/side_hustle/EchoAgent/backend/data
✅ DB path: music_relational.db
Database initialized at: sqlite:///music_relational.db


In [2]:
# Nuke everything to start afresh

from db import init_db, DBSession, Base, Track, Artist, Album, AudioFeature, Lyrics
from sqlalchemy import create_engine

engine = create_engine(DB_URL)
Base.metadata.drop_all(engine)
init_db(DB_URL)
print("🧹 DB reset!")

Database initialized at: sqlite:///music_relational.db
🧹 DB reset!


In [3]:
# Phase 1 - Core track/artist/album/audio feature data from MSD HDF5 files

ingest_h5_dir(Path(H5_DIR), DB_URL)

# Quick check
with DBSession(DB_URL) as s:
    print("Tracks loaded:", s.query(Track).count())

Found 10000 HDF5 files.


HDF5 files: 100%|██████████| 10000/10000 [00:20<00:00, 489.08it/s]

✅ Ingested core metadata + audio features from MSD.
Tracks loaded: 10000


In [4]:
# Counting total tracks/artists/audio features/albums to sanity check ingestion looks reasonable

with DBSession(DB_URL) as s:
    total = s.query(Track).count()
    print(f"✅ {total} tracks, {s.query(Artist).count()} artists")
    print(f"Audio features: {s.query(AudioFeature).count()}")
    print(f"Albums: {s.query(Album).count()}")

✅ 10000 tracks, 3888 artists
Audio features: 10000
Albums: 8061


In [5]:
# Phase 2 - Last.fm tags (JSON subset)

ingest_lastfm_json_subset(Path(LASTFM_JSON_DIR), DB_URL)

with DBSession(DB_URL) as s:
    tags_count = (
        s.query(Track)
        .filter(Track.top_tags_json != 'null')  # SQLite stores JSON null as the string "null"
        .count()
    )
    print(f"Tracks with non-empty tags: {tags_count}")


Found 9330 Last.fm JSON files.


JSON tags: 100%|██████████| 9330/9330 [00:00<00:00, 10491.83it/s]


Loaded 4833 tagged tracks from JSON.


Updating JSON tags: 100%|██████████| 4833/4833 [00:00<00:00, 1222916.94it/s]


✅ Updated 4833 tracks with Last.fm JSON tags.
Tracks with non-empty tags: 4833


In [6]:
# Phase 3 - Lyrics (MXM BoW from SQLite)
ingest_musixmatch_sqlite(Path(MXM_DB), DB_URL)

with DBSession(DB_URL) as s:
    print("Lyrics BoW:", s.query(Lyrics).count())

Loading lyrics from ../../../Data/mxm_dataset.db...


MXM SQLite query:   0%|          | 0/10000 [00:00<?, ?it/s]

Insert BoW: 100%|██████████| 2350/2350 [00:00<00:00, 8973.77it/s]


✅ Ingested 2350 MXM BoW lyrics.
Lyrics BoW: 2350


In [7]:
# Phase 4 - Seed genres
ingest_genres(Path(LASTFM_MAP), DB_URL)

with DBSession(DB_URL) as s:
    print("Tracks with seed_genre:", s.query(Track).filter(Track.seed_genre.isnot(None)).count())


Parsing genres from ../../../Data/msd_lastfm_map.cls...


Last.fm map .cls: 505223it [00:00, 763867.05it/s]


Loaded 505216 genre mappings.


Updating seed_genre: 100%|██████████| 505216/505216 [00:00<00:00, 9189598.38it/s]


✅ Ingested 4833 tracks with seed_genre from tagtraum.
Tracks with seed_genre: 4833


In [8]:
# Randomly sample some tracks with tags to verify they look reasonable

from random import sample

with DBSession(DB_URL) as s:
    tagged = s.query(Track).filter(Track.top_tags_json.isnot(None)).limit(20).all()
    
    print("Sample tagged tracks:")
    for t in sample(tagged, min(10, len(tagged))):
        tags = t.top_tags_json or {}
        genre = t.seed_genre
        print(f"{t.track_id[:12]}... | tags: {list(tags.keys())[:3]} | seed: '{genre}'")


Sample tagged tracks:
TRARROY128F4... | tags: [] | seed: 'None'
TRARRER128F9... | tags: ["['chill', '100']", "['Fusion', '100']"] | seed: 'chill'
TRARRZU128F4... | tags: ["['on the road', '100']", "['Titletracks', '100']"] | seed: 'ontheroad'
TRARUDQ128F9... | tags: [] | seed: 'None'
TRARREF128F4... | tags: ["['punk', '100']", "['halloween', '52']", "['punk rock', '40']"] | seed: 'rock'
TRARRMK12903... | tags: ["['Sludge', '100']", "['doom metal', '100']", "['grindcore', '100']"] | seed: 'freejazz'
TRARRJL128F9... | tags: ["['french', '100']", "['menu du jour', '25']", "['bright voice', '25']"] | seed: 'french'
TRARUOP12903... | tags: ["['metalcore', '100']", "['metal', '81']", "['thrash metal', '62']"] | seed: 'hardcore'
TRARRWA128F4... | tags: ["['post-hardcore', '100']", "['hardcore-punk', '100']"] | seed: 'hardcore'
TRARURM128F9... | tags: [] | seed: 'None'
